<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/ecomarket-agent-solution/blob/main/notebooks/EcoMarket_Agent_Solution.ipynb)

# Maestría en Inteligencia Artificial  
## IA Generativa
---
# EcoMarket AI Support — Proyecto Final: Agente de IA para Devoluciones

**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

## Descripción

Este notebook extiende la arquitectura RAG del Taller Práctico #2 incorporando un **Agente de IA** capaz de ejecutar acciones autónomas sobre el sistema de EcoMarket.

A diferencia del sistema RAG anterior (puramente consultivo), este agente puede:
- **Consultar** el estado de pedidos en tiempo real (tracking, retrasos, entregas).
- **Verificar** si un pedido es elegible para devolución aplicando reglas deterministas.
- **Generar** una etiqueta de devolución con código único y fecha límite de envío.
- **Responder** consultas generales usando la cadena RAG del Taller 2 como herramienta.

El agente implementa un patrón **Router** usando `create_agent` de LangChain: analiza la intención del usuario y decide qué herramienta invocar.

### Stack tecnológico
| Componente | Librería |
|---|---|
| Agente | `langchain.agents.create_agent` (LangGraph CompiledStateGraph) |
| Herramientas | `langchain_core.tools.tool` |
| LLM | `langchain_groq.ChatGroq` (llama-3.3-70b-versatile) |
| Embeddings | `intfloat/multilingual-e5-large` (HuggingFace) |
| Vector Store | ChromaDB via `langchain_chroma` |
| RAG Chain | `langchain_classic.chains.RetrievalQA` |
| Interfaz | Gradio `gr.ChatInterface` |

## 1. Configuración del Entorno

In [141]:
import warnings
from importlib import metadata

warnings.filterwarnings('ignore')

installed_packages = {dist.metadata['Name'].lower() for dist in metadata.distributions() if dist.metadata.get('Name')}
IN_COLAB = 'google-colab' in installed_packages
print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'}")

Entorno: Local


In [142]:
if IN_COLAB:
    !wget -q https://raw.githubusercontent.com/sebastianb92/ecomarket-agent-solution/main/requirements.txt -O requirements.txt
    !uv pip install -r requirements.txt
else:
    print('Entorno local — instala con: pip install -r requirements.txt')

Entorno local — instala con: pip install -r requirements.txt


## 2. Importaciones

In [143]:
# Librerías estándar
import os
import uuid
import json
import hashlib
from pathlib import Path
from datetime import datetime, timedelta

# Datos
import pandas as pd

# Entorno
if IN_COLAB:
    from google.colab import userdata
else:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(), override=True)

# Configurar HF_TOKEN para evitar errores 429 en la API de HuggingFace
hf_token = userdata.get('HF_TOKEN') if IN_COLAB else os.getenv('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

# LLM
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

# Embeddings y vector store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata

# Carga de documentos
from langchain_community.document_loaders import (
    DataFrameLoader, PyPDFLoader, JSONLoader
)

# Procesamiento de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Prompts y cadenas (RAG — Taller 2)
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import RetrievalQA

# Agente
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

print('Importaciones completadas correctamente.')
if hf_token:
    print('HF_TOKEN configurado correctamente.')
else:
    print('⚠️ HF_TOKEN no encontrado. Puede haber errores 429 al descargar modelos.')

Importaciones completadas correctamente.
HF_TOKEN configurado correctamente.


## 3. Carga de Datos

In [173]:
if IN_COLAB:
    DATA_DIR = Path('data')
    DATA_DIR.mkdir(exist_ok=True)

    ![ ! -f data/FAQ.json ] && wget -q -O data/FAQ.json \
    https://raw.githubusercontent.com/sebastianb92/ecomarket-agent-solution/main/data/FAQ.json

    ![ ! -f data/politica_devoluciones.pdf ] && wget -q -O data/politica_devoluciones.pdf \
    "https://raw.githubusercontent.com/sebastianb92/ecomarket-agent-solution/main/data/POL%C3%8DTICA%20DE%20DEVOLUCIONES.pdf"

    ![ ! -f data/pedidos_ecomarket.xlsx ] && wget -q -O data/pedidos_ecomarket.xlsx \
    https://raw.githubusercontent.com/sebastianb92/ecomarket-agent-solution/main/data/pedidos_ecomarket.xlsx

    print('Datos descargados desde GitHub')
else:
    DATA_DIR = Path('../data')
    print(f'Datos locales en: {DATA_DIR.resolve()}')

df_pedidos = pd.read_excel(DATA_DIR / 'pedidos_ecomarket.xlsx')
print(f'Pedidos cargados: {len(df_pedidos)} registros')
print(f'Columnas: {df_pedidos.columns.tolist()}')

Datos locales en: /Users/gome33773/Documents/MIAA/IAGenerativa/Entregas/ecomarket-agent-solution-plus/data
Pedidos cargados: 30 registros
Columnas: ['pedido_id', 'cliente', 'estado', 'producto', 'fecha_pedido', 'entrega_estimada', 'entrega_real', 'transportista', 'tracking_url', 'notas']


## 4. Inicialización del LLM, Embeddings y Vector Store

Configura el **LLM**. Cambia `LLM_PROVIDER` para alternar entre proveedores:
- `"ollama"` — modelo local sin límites (requiere Ollama corriendo en localhost)
- `"gemini"` — Google Gemini Flash, 15 req/min gratis
- `"groq"` — Groq Cloud, más rápido pero con rate limits estrictos en free tier

> **Nota:** `ollama` no está disponible en Google Colab (no hay servidor local). En Colab usa `"gemini"` o `"groq"`.

In [174]:
# ============================================================
# CONFIGURACIÓN: Cambia esta variable para alternar proveedor
# Opciones: "ollama", "gemini", "groq"
# ============================================================
LLM_PROVIDER = "groq"

if LLM_PROVIDER == "ollama":
    if IN_COLAB:
        raise RuntimeError("Ollama no está disponible en Colab. Usa 'gemini' o 'groq'.")
    llm = ChatOllama(
        model='llama3.2:3b',
        temperature=0.3
    )
    print(f'LLM inicializado: Ollama ({llm.model}) — sin rate limits')

elif LLM_PROVIDER == "gemini":
    google_api_key = userdata.get('GOOGLE_API_KEY') if IN_COLAB else os.getenv('GOOGLE_API_KEY')
    llm = ChatGoogleGenerativeAI(
        model='gemini-2.0-flash',
        google_api_key=google_api_key,
        temperature=0.3
    )
    print(f'LLM inicializado: Google Gemini ({llm.model}) — 15 req/min')

elif LLM_PROVIDER == "groq":
    api_key = userdata.get('GROQ_API_KEY') if IN_COLAB else os.getenv('GROQ_API_KEY')
    llm = ChatGroq(
        model='llama-3.3-70b-versatile',
        api_key=api_key,
        temperature=0.3
    )
    print(f'LLM inicializado: Groq ({llm.model_name}) — rate limits estrictos')

else:
    raise ValueError(f"LLM_PROVIDER inválido: '{LLM_PROVIDER}'. Usa 'ollama', 'gemini' o 'groq'.")

LLM inicializado: Groq (llama-3.3-70b-versatile) — rate limits estrictos


Inicializa el modelo de **embeddings** para representar texto como vectores.

In [175]:
device = 'cuda' if IN_COLAB else 'cpu'

embeddings = HuggingFaceEmbeddings(
    model_name='intfloat/multilingual-e5-large',
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)
print('Embeddings inicializados.')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 18998.32it/s]


Embeddings inicializados.


Crea y configura la **vector store** para almacenar y consultar embeddings.

In [176]:
vector_store = Chroma(
    collection_name='ecomarket_agent_collection',
    embedding_function=embeddings,
    persist_directory='./chroma_agent_db',
)
print('Vector store inicializado.')

Vector store inicializado.


## 5. Indexación de Documentos

In [177]:
# Excel
df_pedidos['contenido'] = df_pedidos.apply(lambda row: ' | '.join(str(v) for v in row), axis=1)
docs_excel = DataFrameLoader(df_pedidos, page_content_column='contenido').load()

# PDF
docs_pdf = PyPDFLoader(str(DATA_DIR / 'POLÍTICA DE DEVOLUCIONES.pdf')).load()

# JSON
docs_json = JSONLoader(
    file_path=str(DATA_DIR / 'FAQ.json'),
    jq_schema='.faq[] | "Categoría: \(.categoria)\\nPregunta: \(.pregunta)\\nRespuesta: \(.respuesta)"',
    text_content=True
).load()

docs = docs_excel + docs_pdf + docs_json
print(f'Documentos cargados: {len(docs)} (Excel: {len(docs_excel)}, PDF: {len(docs_pdf)}, JSON: {len(docs_json)})')

Documentos cargados: 54 (Excel: 30, PDF: 4, JSON: 20)


In [178]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)
all_splits = filter_complex_metadata(all_splits)
document_ids = vector_store.add_documents(documents=all_splits)
print(f'Sub-documentos indexados: {len(all_splits)}')

Sub-documentos indexados: 55


## 6. Cadena RAG (RetrievalQA) - Recuperación y generación

Configura el **retriever** para buscar los 3 documentos más similares en la base vectorial.

In [179]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

Este prompt define a EcoBot, un asistente de e-commerce que responde consultas sobre estado de pedidos y devoluciones usando información de un contexto (RAG).

Primero clasifica la intención del usuario y luego aplica un flujo específico. Incluye una regla crítica: solo usa información de pedidos si el usuario proporciona explícitamente un número de pedido, evitando errores comunes de asociación automática entre productos y pedidos.

In [180]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres EcoBot, asistente de e-commerce de EcoMarket (productos sostenibles).

REGLAS:
- Usa SOLO el contexto proporcionado
- NO inventes información
- Si no encuentras datos, dilo claramente
- Responde SIEMPRE en español
- Tono amable, claro y profesional
- Sé conciso pero útil

REGLA CRÍTICA:
- SOLO usa información de pedidos si el usuario menciona explícitamente un número de pedido
- PROHIBIDO inferir pedidos a partir del producto mencionado
- Si el usuario solo menciona un producto sin número de pedido, responde como consulta general

Siempre cierra con: ¿Puedo ayudarte con algo más?"""),
    ("human", """Contexto:\n{context}\n\nPregunta:\n{question}""")
])

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": rag_prompt}
)
print('Cadena RAG (RetrievalQA) configurada.')

Cadena RAG (RetrievalQA) configurada.


## 7. Sistema de Registro de Acciones (Action Log)

Implementa la Capa 1 del sistema de monitoreo propuesto en la Fase 3. Cada invocación de una herramienta queda registrada en `logs/agent_actions.jsonl`.

In [182]:
LOG_PATH = Path('logs')
LOG_PATH.mkdir(exist_ok=True)

SESSION_ID = str(uuid.uuid4())[:8]

def _registrar_accion(herramienta: str, inputs: dict, outputs: dict) -> None:
    """Registra una acción del agente en el log JSONL."""
    entrada = {
        'timestamp': datetime.now().isoformat(),
        'session_id': SESSION_ID,
        'herramienta': herramienta,
        'input': inputs,
        'output': outputs
    }
    with open(LOG_PATH / 'agent_actions.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(entrada, ensure_ascii=False) + '\n')

print(f'Session ID: {SESSION_ID}')
print(f"Log en: {(LOG_PATH / 'agent_actions.jsonl').resolve()}")

Session ID: 9ea0f9ba
Log en: /Users/gome33773/Documents/MIAA/IAGenerativa/Entregas/ecomarket-agent-solution-plus/notebooks/logs/agent_actions.jsonl


## 8. Definición de Herramientas del Agente

El decorator `@tool` de `langchain_core.tools` convierte cada función en una herramienta que el agente puede invocar. El docstring es la descripción que el LLM lee para decidir cuándo usar cada herramienta.

### Herramientas:
1. `consultar_estado_pedido` — Consulta estado de un pedido
2. `verificar_elegibilidad_devolucion` — Verifica si un producto puede ser devuelto
3. `generar_etiqueta_devolucion` — Genera etiqueta de envío para devolución
4. `consultar_base_conocimiento` — Consulta el RAG para preguntas generales

### Herramienta 1: `consultar_estado_pedido`

Consulta el DataFrame de pedidos en tiempo real para obtener toda la información disponible de un pedido.

In [183]:
@tool
def consultar_estado_pedido(numero_pedido: str) -> str:
    """Consulta el estado actual de un pedido en la base de datos de EcoMarket.
    Usa esta herramienta cuando el usuario pregunte por el estado, tracking o
    información de un pedido específico SIN intención de devolverlo.
    Requiere el número de pedido (formato ECO-XXXXX).
    Args:
        numero_pedido: Identificador del pedido, por ejemplo ECO-12345.
    """
    numero_pedido = numero_pedido.strip().upper()
    fila = df_pedidos[df_pedidos['pedido_id'].str.upper() == numero_pedido]

    if fila.empty:
        resultado = {
            'encontrado': False,
            'mensaje': f'No se encontró el pedido {numero_pedido} en la base de datos.',
            'sugerencia': 'Verifica el número de pedido. Contacto: soporte@ecomarket.com o +34 900 123 456.'
        }
        _registrar_accion('consultar_estado_pedido', {'numero_pedido': numero_pedido}, resultado)
        return json.dumps(resultado, ensure_ascii=False)

    pedido = fila.iloc[0]
    info = {'encontrado': True, 'numero_pedido': numero_pedido}
    for col in pedido.index:
        if col != 'contenido' and pd.notna(pedido[col]):
            info[str(col).lower().replace(' ', '_')] = str(pedido[col])

    _registrar_accion('consultar_estado_pedido', {'numero_pedido': numero_pedido}, info)
    return json.dumps(info, ensure_ascii=False)

### Herramienta 2: `verificar_elegibilidad_devolucion`

Aplica reglas **deterministas** de elegibilidad basadas en la política oficial de EcoMarket. La decisión no depende del LLM, lo que garantiza consistencia y previene alucinaciones.

In [184]:
# Registro de devoluciones aprobadas (previene bypass del flujo)
devoluciones_aprobadas = {}

# Categorías de productos no elegibles
PRODUCTOS_HIGIENE = ['jabón', 'shampoo', 'champú', 'desodorante', 'crema', 'gel', 'pasta dental']
PRODUCTOS_PERECEDEROS = ['frutos secos', 'alimento', 'comida', 'snack', 'orgánico comestible', 'té', 'infusión']

@tool
def verificar_elegibilidad_devolucion(pedido_id: str, motivo: str) -> str:
    """Verifica si un pedido de EcoMarket es elegible para devolución.
    Úsala cuando el usuario quiera DEVOLVER un producto y proporcione un número de pedido (ej. ECO-12345).
    Requiere el número de pedido Y el motivo de la devolución.
    Args:
        pedido_id: Identificador del pedido, por ejemplo ECO-12345.
        motivo: Razón de la devolución proporcionada por el usuario.
    """
    pedido_id = pedido_id.strip().upper()
    motivo = motivo.strip().lower()
    fila = df_pedidos[df_pedidos['pedido_id'].str.upper() == pedido_id]

    if fila.empty:
        resultado = {
            'pedido_id': pedido_id, 'elegible': False,
            'mensaje': f'No se encontró el pedido {pedido_id}. Verifica el número e intenta nuevamente.'
        }
        _registrar_accion('verificar_elegibilidad_devolucion', {'pedido_id': pedido_id, 'motivo': motivo}, resultado)
        return json.dumps(resultado, ensure_ascii=False)

    pedido = fila.iloc[0]
    estado = pedido['estado']
    cliente = pedido['cliente']
    producto = pedido['producto']
    fecha_entrega = pedido.get('entrega_real', None)
    fecha_pedido = pedido.get('fecha_pedido', None)

    # --- Verificación 1: Estado del pedido ---
    ESTADOS_ELEGIBLES = {'ENTREGADO', 'LISTO PARA RECOGIDA'}
    RAZONES_NO_ELEGIBLE = {
        'EN TRÁNSITO':        'el pedido aún está EN TRÁNSITO y no ha sido recibido.',
        'RETRASADO':          'el pedido está RETRASADO y aún no ha sido entregado.',
        'PROCESANDO':         'el pedido está siendo PROCESADO y no ha salido del almacén.',
        'CANCELADO':          'el pedido fue CANCELADO. Si no recibiste reembolso, contacta soporte@ecomarket.com.',
        'PENDIENTE DE PAGO':  'el pedido está PENDIENTE DE PAGO y no ha sido confirmado.',
        'RETENIDO EN ADUANA': 'el pedido está RETENIDO EN ADUANA y no está bajo control de EcoMarket.',
        'DEVUELTO':           'el pedido ya fue DEVUELTO previamente.',
    }

    if estado not in ESTADOS_ELEGIBLES:
        razon = RAZONES_NO_ELEGIBLE.get(estado, f'el estado actual es {estado}.')
        resultado = {
            'pedido_id': pedido_id, 'cliente': cliente, 'producto': producto,
            'estado': estado, 'elegible': False,
            'mensaje': f'El pedido {pedido_id} ({producto}) NO es elegible para devolución porque {razon}'
        }
        _registrar_accion('verificar_elegibilidad_devolucion', {'pedido_id': pedido_id, 'motivo': motivo}, resultado)
        return json.dumps(resultado, ensure_ascii=False)

    # --- Verificación 2: Daño en tránsito (siempre se aprueba con compensación) ---
    KEYWORDS_DANO = ['dañado', 'roto', 'aplastado', 'defectuoso', 'golpeado', 'daño', 'golpe']
    if any(kw in motivo for kw in KEYWORDS_DANO):
        devoluciones_aprobadas[pedido_id] = {
            'producto': producto, 'cliente': cliente, 'motivo': motivo, 'tipo': 'defecto_transito'
        }
        resultado = {
            'pedido_id': pedido_id, 'cliente': cliente, 'producto': producto,
            'estado': estado, 'elegible': True,
            'mensaje': f'Devolución aprobada automáticamente por daño en tránsito. EcoMarket cubre todos los costos de envío.',
            'compensacion_adicional': 'Cupón de descuento del 10% para próxima compra.',
            'costo_envio': 'Gratuito (cubierto por EcoMarket)'
        }
        _registrar_accion('verificar_elegibilidad_devolucion', {'pedido_id': pedido_id, 'motivo': motivo}, resultado)
        return json.dumps(resultado, ensure_ascii=False)

    # --- Verificación 3: Categoría del producto ---
    producto_lower = producto.lower()
    for cat in PRODUCTOS_HIGIENE:
        if cat in producto_lower:
            resultado = {
                'pedido_id': pedido_id, 'cliente': cliente, 'producto': producto,
                'estado': estado, 'elegible': False,
                'mensaje': f'Los productos de higiene personal ({producto}) no son elegibles para devolución una vez abiertos.',
                'alternativa': 'Contacta soporte@ecomarket.com para opciones de crédito en tienda.'
            }
            _registrar_accion('verificar_elegibilidad_devolucion', {'pedido_id': pedido_id, 'motivo': motivo}, resultado)
            return json.dumps(resultado, ensure_ascii=False)

    for cat in PRODUCTOS_PERECEDEROS:
        if cat in producto_lower:
            resultado = {
                'pedido_id': pedido_id, 'cliente': cliente, 'producto': producto,
                'estado': estado, 'elegible': False,
                'mensaje': f'Los productos perecederos ({producto}) no son elegibles por razones sanitarias.',
                'alternativa': 'Si llegó en mal estado, contacta soporte@ecomarket.com para reembolso especial.'
            }
            _registrar_accion('verificar_elegibilidad_devolucion', {'pedido_id': pedido_id, 'motivo': motivo}, resultado)
            return json.dumps(resultado, ensure_ascii=False)

    # --- Verificación 4: Plazo de 30 días ---
    fecha_ref = fecha_entrega if pd.notna(fecha_entrega) else fecha_pedido
    if pd.notna(fecha_ref):
        try:
            fecha = pd.to_datetime(fecha_ref)
            dias_transcurridos = (datetime.now() - fecha).days
            if dias_transcurridos > 30:
                resultado = {
                    'pedido_id': pedido_id, 'cliente': cliente, 'producto': producto,
                    'estado': estado, 'elegible': False,
                    'mensaje': f'El plazo de devolución de 30 días ha expirado ({dias_transcurridos} días transcurridos).',
                    'alternativa': 'Contacta soporte@ecomarket.com para evaluar excepciones o crédito en tienda.'
                }
                _registrar_accion('verificar_elegibilidad_devolucion', {'pedido_id': pedido_id, 'motivo': motivo}, resultado)
                return json.dumps(resultado, ensure_ascii=False)
        except (ValueError, TypeError):
            pass

    # --- Si pasa todas las verificaciones: APROBADA ---
    devoluciones_aprobadas[pedido_id] = {
        'producto': producto, 'cliente': cliente, 'motivo': motivo, 'tipo': 'devolucion_estandar'
    }
    fecha_str = str(fecha_entrega.date()) if pd.notna(fecha_entrega) else 'fecha no registrada'
    resultado = {
        'pedido_id': pedido_id, 'cliente': cliente, 'producto': producto,
        'estado': estado, 'elegible': True, 'fecha_entrega': fecha_str,
        'mensaje': f'El pedido {pedido_id} ({producto}) es ELEGIBLE para devolución. Entregado: {fecha_str}.',
        'plazo_reembolso': '5-7 días hábiles tras recibir el producto en almacén.',
        'opcion_credito': 'Puedes optar por crédito en tienda con 5% adicional de bonificación.',
        'costo_envio': 'Tarifa plana: 3.95€ (devolución por decisión del cliente).'
    }
    _registrar_accion('verificar_elegibilidad_devolucion', {'pedido_id': pedido_id, 'motivo': motivo}, resultado)
    return json.dumps(resultado, ensure_ascii=False)

### Herramienta 3: `generar_etiqueta_devolucion`

Genera la etiqueta de devolución. **Verifica internamente** que exista una devolución aprobada, creando una dependencia técnica que no puede ser eludida por el LLM.

In [186]:
@tool
def generar_etiqueta_devolucion(pedido_id: str) -> str:
    """Genera una etiqueta de devolución (código, fecha límite, centro de envío) para un pedido elegible.
    SOLO llama esta herramienta DESPUÉS de verificar elegibilidad con verificar_elegibilidad_devolucion.
    La herramienta rechazará la operación si no existe una devolución aprobada previamente.
    Args:
        pedido_id: Identificador del pedido elegible, por ejemplo ECO-12347.
    """
    pedido_id = pedido_id.strip().upper()

    # Verificar que la devolución fue aprobada previamente
    if pedido_id not in devoluciones_aprobadas:
        resultado = {
            'pedido_id': pedido_id, 'exito': False,
            'mensaje': f'No se puede generar etiqueta: no hay devolución aprobada para {pedido_id}. Verifica elegibilidad primero.'
        }
        _registrar_accion('generar_etiqueta_devolucion', {'pedido_id': pedido_id}, resultado)
        return json.dumps(resultado, ensure_ascii=False)

    info_devolucion = devoluciones_aprobadas[pedido_id]
    codigo_devolucion = f"DEV-{hashlib.md5(f'{pedido_id}{datetime.now().isoformat()}'.encode()).hexdigest()[:8].upper()}"
    fecha_limite = (datetime.now() + timedelta(days=14)).strftime('%Y-%m-%d')

    # Transportista según tipo de devolución
    if info_devolucion.get('tipo') == 'defecto_transito':
        transportista = 'DHL Express (recogida a domicilio gratuita)'
        costo = 'GRATUITO — cubierto por EcoMarket'
    else:
        transportista = 'Correos — Punto de recogida más cercano'
        costo = 'Tarifa plana: 3.95€ (se descontará del reembolso)'

    resultado = {
        'pedido_id': pedido_id,
        'cliente': info_devolucion.get('cliente', 'N/A'),
        'producto': info_devolucion.get('producto', 'N/A'),
        'exito': True,
        'codigo_devolucion': codigo_devolucion,
        'transportista': transportista,
        'costo_envio': costo,
        'direccion_almacen': 'Centro de Devoluciones EcoMarket — C/ Sostenibilidad 42, Nave 7, 28042 Madrid',
        'fecha_limite_envio': fecha_limite,
        'instrucciones': [
            '1. Empaqueta el producto en su embalaje original (si es posible).',
            '2. Imprime esta etiqueta y pégala en el exterior del paquete.',
            f'3. Entrega en {transportista.split("—")[0].strip() if "—" in transportista else transportista} antes del {fecha_limite}.',
            f'4. Código de seguimiento: {codigo_devolucion}.',
            '5. El reembolso se procesará en 5-7 días hábiles tras recepción.'
        ],
        'mensaje': f'Etiqueta de devolución generada exitosamente para el pedido {pedido_id}.'
    }
    _registrar_accion('generar_etiqueta_devolucion', {'pedido_id': pedido_id}, resultado)
    return json.dumps(resultado, ensure_ascii=False)

### Herramienta 4: `consultar_base_conocimiento`

Encapsula la cadena RAG del Taller 2 usando RetrievalQA. El agente la invoca para preguntas generales sobre política, FAQ o consultas de estado sin intención de devolución.

In [187]:
@tool
def consultar_base_conocimiento(pregunta: str) -> str:
    """Responde preguntas generales sobre política de devoluciones, estado de pedidos y FAQ de EcoMarket.
    Úsala para consultas que NO requieren ejecutar una acción concreta (verificar o generar etiqueta).
    También úsala cuando el usuario pregunta por el estado de un pedido sin intención de devolverlo
    y la herramienta consultar_estado_pedido no tiene la información necesaria.
    Args:
        pregunta: La consulta del usuario en lenguaje natural.
    """
    try:
        docs_relevantes = retriever.invoke(pregunta)
        if not docs_relevantes:
            return 'No se encontró información relevante en la base de conocimiento.'
        contexto = '\n\n---\n\n'.join([doc.page_content for doc in docs_relevantes])
        _registrar_accion('consultar_base_conocimiento', {'pregunta': pregunta[:100]}, {'docs_recuperados': len(docs_relevantes)})
        return f'Información recuperada de la base de conocimiento de EcoMarket:\n\n{contexto}'
    except Exception as e:
        return f'Error al consultar la base de conocimiento: {str(e)}'

In [188]:
tools = [consultar_estado_pedido, verificar_elegibilidad_devolucion, generar_etiqueta_devolucion, consultar_base_conocimiento]
print(f"Herramientas registradas: {[t.name for t in tools]}")

Herramientas registradas: ['consultar_estado_pedido', 'verificar_elegibilidad_devolucion', 'generar_etiqueta_devolucion', 'consultar_base_conocimiento']


## 9. Inicialización del Agente

`create_agent` de LangChain proporciona una implementación de agente ReAct lista para producción. El agente ejecuta herramientas en bucle para lograr un objetivo, razonando paso a paso sobre qué herramienta usar.

In [189]:
AGENT_SYSTEM_PROMPT = """
Eres EcoBot, el asistente virtual de atención al cliente de EcoMarket,
una tienda de productos sostenibles y ecológicos.

## ROL Y TONO
Actúas como un agente de soporte empático, claro y honesto. Tu tono es
cálido pero profesional: como un asesor que realmente quiere resolver el
problema, no solo cerrar el ticket. Usa frases cortas, evita tecnicismos
y adapta tu nivel de detalle a lo que el usuario necesita.

## HERRAMIENTAS Y CUÁNDO USARLAS

### `consultar_estado_pedido`
- USA cuando: el usuario quiere saber el estado, tracking o información
  de un pedido Y NO menciona devolución.
- NO USES si: el usuario quiere devolver un producto.

### `verificar_elegibilidad_devolucion`
- USA cuando: el usuario quiere devolver un producto Y ya proporcionó el
  número de pedido.
- NO USES si: el usuario no ha dado el número de pedido (pídelo primero).
- NO USES si: el usuario no ha dado un motivo (pregúntalo primero).

### `generar_etiqueta_devolucion`
- USA cuando: verificaste elegibilidad con resultado ELEGIBLE.
- NUNCA antes de verificar elegibilidad.
- Procede automáticamente si la devolución fue aprobada.

### `consultar_base_conocimiento`
- USA cuando: preguntas generales sobre políticas, envíos, productos,
  o cualquier duda que no sea una solicitud de acción concreta.
- NUNCA USES para consultas que no tengan relación con EcoMarket.

## FLUJO PARA DEVOLUCIONES (sigue este orden estrictamente)
1. Usuario menciona devolución → pregunta el número de pedido si no lo tiene.
2. Pregunta el motivo si no lo mencionó.
3. Tienes pedido + motivo → llama a `verificar_elegibilidad_devolucion`.
4. Resultado ELEGIBLE → llama a `generar_etiqueta_devolucion` y entrega
   instrucciones claras con todos los datos de la etiqueta.
5. Resultado NO ELEGIBLE → explica la razón con empatía, sin culpar al
   usuario, y ofrece la alternativa indicada por la herramienta.

## CONSULTAS DE CONTACTO Y SOPORTE
Si el usuario pregunta por teléfonos, correos o formas de contacto,
responde directamente sin invocar ninguna herramienta:
- Email: soporte@ecomarket.com
- Teléfono: +34 900 123 456 (Lunes a Viernes, 9:00 - 18:00)
- Chat en vivo: www.ecomarket.com/chat

## CONSULTAS FUERA DE DOMINIO
Si la consulta NO tiene relación con pedidos, devoluciones, productos o
servicios de EcoMarket:
- NO invoques NINGUNA herramienta.
- Responde directamente que solo puedes ayudar con temas de EcoMarket.
- Ofrece las opciones de contacto.
Ejemplos de consultas fuera de dominio: recetas, deportes, clima, matemáticas,
programación, historia, o cualquier tema ajeno a EcoMarket.

## REGLAS ESTRICTAS
- Responde SIEMPRE en español.
- NUNCA inventes números de pedido, fechas, estados ni datos de contacto.
- Si una herramienta falla, infórmalo con honestidad y sugiere contactar
  soporte humano.
- No repitas información que el usuario ya te dio.
- Usa el nombre del cliente cuando esté disponible.
- Para devoluciones aprobadas, presenta las instrucciones en pasos numerados.
- Para rechazos, primero muestra empatía, luego la razón, luego alternativas.
- Al finalizar, cierra con: ¿Hay algo más en lo que pueda ayudarte?
""".strip()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=AGENT_SYSTEM_PROMPT
)

print('Agente inicializado correctamente.')
print(f'Herramientas disponibles: {[t.name for t in tools]}')
print(f'Tipo de agente: {type(agent).__name__}')

Agente inicializado correctamente.
Herramientas disponibles: ['consultar_estado_pedido', 'verificar_elegibilidad_devolucion', 'generar_etiqueta_devolucion', 'consultar_base_conocimiento']
Tipo de agente: CompiledStateGraph


### Función auxiliar `preguntar_al_agente`

In [190]:
def preguntar_al_agente(consulta: str, verbose: bool = True) -> str:
    """Invoca el agente y retorna su respuesta final en texto."""
    try:
        result = agent.invoke({"messages": [HumanMessage(content=consulta)]})
        mensajes = result["messages"]

        if verbose:
            print("\n--- Pasos intermedios ---")
            for msg in mensajes:
                tipo = type(msg).__name__
                print(f"  [{tipo}]: {str(msg.content)[:200]}")
            print("--- Fin pasos intermedios ---\n")

        # Buscar el AIMessage con contenido más rico
        respuesta_final = ""
        for msg in reversed(mensajes):
            if hasattr(msg, "content") and len(str(msg.content).strip()) > 30:
                respuesta_final = str(msg.content)
                break

        if not respuesta_final:
            respuesta_final = str(mensajes[-1].content)

        return respuesta_final

    except Exception as e:
        return f"Error del agente: {str(e)}"

## 10. Evaluación del Comportamiento del Agente

Se prueban 9 escenarios para verificar que el agente:
1. Invoca las herramientas correctas según la intención.
2. Maneja pedidos elegibles (flujo completo: verificar → generar etiqueta).
3. Maneja pedidos inelegibles con empatía y alternativas.
4. Responde consultas generales usando solo el RAG.
5. Maneja pedidos inexistentes y consultas fuera de dominio.

### Prueba 1 — Consulta de estado de pedido (EN TRÁNSITO)

In [160]:
respuesta = preguntar_al_agente(
    '¿Cuál es el estado de mi pedido ECO-12345?'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)


--- Pasos intermedios ---
  [HumanMessage]: ¿Cuál es el estado de mi pedido ECO-12345?
  [AIMessage]: 
  [ToolMessage]: {"encontrado": true, "numero_pedido": "ECO-12345", "pedido_id": "ECO-12345", "cliente": "María García", "estado": "EN TRÁNSITO", "producto": "Kit de bambú reutilizable (3 piezas)", "fecha_pedido": "20
  [AIMessage]: Tu pedido ECO-12345 está en tránsito y se encuentra en el centro logístico de Madrid. La entrega está estimada para el 5 de julio de 2024. Puedes seguir el progreso de tu pedido en el siguiente enlace
--- Fin pasos intermedios ---


RESPUESTA FINAL:
Tu pedido ECO-12345 está en tránsito y se encuentra en el centro logístico de Madrid. La entrega está estimada para el 5 de julio de 2024. Puedes seguir el progreso de tu pedido en el siguiente enlace: https://track.dhl.com/ECO12345

¿Hay algo más en lo que pueda ayudarte?


### Prueba 2 — Devolución con pedido ELEGIBLE (flujo completo)

In [191]:
respuesta = preguntar_al_agente(
    'Quiero devolver mi pedido ECO-12347, la botella no es del tamaño que esperaba.'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)


--- Pasos intermedios ---
  [HumanMessage]: Quiero devolver mi pedido ECO-12347, la botella no es del tamaño que esperaba.
  [AIMessage]: 
  [ToolMessage]: {"pedido_id": "ECO-12347", "cliente": "Ana Martínez", "producto": "Botella de acero inoxidable 750ml", "estado": "ENTREGADO", "elegible": true, "fecha_entrega": "2026-05-08", "mensaje": "El pedido ECO
  [AIMessage]: 
  [ToolMessage]: {"pedido_id": "ECO-12347", "cliente": "Ana Martínez", "producto": "Botella de acero inoxidable 750ml", "exito": true, "codigo_devolucion": "DEV-A6893160", "transportista": "Correos — Punto de recogida
  [AIMessage]: ¡Hola Ana! Tu pedido ECO-12347 es elegible para devolución. La botella de acero inoxidable 750ml fue entregada el 2026-05-08. Para proceder con la devolución, te proporciono la siguiente información:

--- Fin pasos intermedios ---


RESPUESTA FINAL:
¡Hola Ana! Tu pedido ECO-12347 es elegible para devolución. La botella de acero inoxidable 750ml fue entregada el 2026-05-08. Para proceder con

### Prueba 3 — Devolución con pedido NO ELEGIBLE (estado: EN TRÁNSITO)

In [162]:
respuesta = preguntar_al_agente(
    'Quiero devolver mi pedido ECO-12345, ya no lo necesito.'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)


--- Pasos intermedios ---
  [HumanMessage]: Quiero devolver mi pedido ECO-12345, ya no lo necesito.
  [AIMessage]: 
  [ToolMessage]: {"pedido_id": "ECO-12345", "cliente": "María García", "producto": "Kit de bambú reutilizable (3 piezas)", "estado": "EN TRÁNSITO", "elegible": false, "mensaje": "El pedido ECO-12345 (Kit de bambú reut
  [AIMessage]: Lo siento mucho, María. Entiendo que ya no necesitas el Kit de bambú reutilizable que pediste. Sin embargo, según nuestras políticas, el pedido ECO-12345 no es elegible para devolución en este momento
--- Fin pasos intermedios ---


RESPUESTA FINAL:
Lo siento mucho, María. Entiendo que ya no necesitas el Kit de bambú reutilizable que pediste. Sin embargo, según nuestras políticas, el pedido ECO-12345 no es elegible para devolución en este momento porque aún está en tránsito y no ha sido recibido. 

Te recomiendo esperar a que recibas el pedido y, si sigues deseando devolverlo, podemos volver a evaluar la elegibilidad para la devolución. ¿Hay 

### Prueba 4 — Devolución con pedido CANCELADO

In [163]:
respuesta = preguntar_al_agente(
    'Necesito hacer la devolución del pedido ECO-12349. No me gustó el producto.'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)


--- Pasos intermedios ---
  [HumanMessage]: Necesito hacer la devolución del pedido ECO-12349. No me gustó el producto.
  [AIMessage]: 
  [ToolMessage]: {"pedido_id": "ECO-12349", "cliente": "Sofia Ruiz", "producto": "Cepillo de dientes de bambú (pack x4)", "estado": "CANCELADO", "elegible": false, "mensaje": "El pedido ECO-12349 (Cepillo de dientes d
  [AIMessage]: Lo siento mucho, Sofia. Entiendo que deseabas devolver el pedido ECO-12349 porque no te gustó el producto. Sin embargo, he verificado el estado de tu pedido y resulta que el pedido ECO-12349 fue cance
--- Fin pasos intermedios ---


RESPUESTA FINAL:
Lo siento mucho, Sofia. Entiendo que deseabas devolver el pedido ECO-12349 porque no te gustó el producto. Sin embargo, he verificado el estado de tu pedido y resulta que el pedido ECO-12349 fue cancelado. Desafortunadamente, esto significa que no es elegible para devolución. 

Si no has recibido el reembolso correspondiente, te recomiendo contactar directamente con nuestro equ

### Prueba 5 — Devolución APROBADA por daño en tránsito

In [192]:
respuesta = preguntar_al_agente(
    'Mi pedido ECO-12348 llegó con el paquete completamente roto y aplastado. Quiero devolverlo.'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)


--- Pasos intermedios ---
  [HumanMessage]: Mi pedido ECO-12348 llegó con el paquete completamente roto y aplastado. Quiero devolverlo.
  [AIMessage]: 
  [ToolMessage]: {"pedido_id": "ECO-12348", "cliente": "Luis Fernández", "producto": "Set de cubiertos de bambú", "estado": "ENTREGADO", "elegible": true, "mensaje": "Devolución aprobada automáticamente por daño en tr
  [AIMessage]: 
  [ToolMessage]: {"pedido_id": "ECO-12348", "cliente": "Luis Fernández", "producto": "Set de cubiertos de bambú", "exito": true, "codigo_devolucion": "DEV-F90DFDDF", "transportista": "DHL Express (recogida a domicilio
  [AIMessage]: ¡Hola Luis! Lamentamos mucho que tu pedido haya llegado dañado. La devolución ha sido aprobada automáticamente. Para proceder, te hemos generado una etiqueta de devolución gratuita con DHL Express. Po
--- Fin pasos intermedios ---


RESPUESTA FINAL:
¡Hola Luis! Lamentamos mucho que tu pedido haya llegado dañado. La devolución ha sido aprobada automáticamente. Para proceder, te 

### Prueba 6 — Consulta general de política (sin número de pedido)

In [172]:
respuesta = preguntar_al_agente(
    '¿Cuántos días tengo para hacer una devolución?'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)


--- Pasos intermedios ---
  [HumanMessage]: ¿Cuántos días tengo para hacer una devolución?
  [AIMessage]: 
  [ToolMessage]: Información recuperada de la base de conocimiento de EcoMarket:

Categoría: Devoluciones
Pregunta: ¿Cuál es el plazo para devolver un producto?
Respuesta: Puedes solicitar una devolución dentro de los
  [AIMessage]: Puedes solicitar una devolución dentro de los 30 días naturales desde la fecha de entrega. ¿Hay algo más en lo que pueda ayudarte?
--- Fin pasos intermedios ---


RESPUESTA FINAL:
Puedes solicitar una devolución dentro de los 30 días naturales desde la fecha de entrega. ¿Hay algo más en lo que pueda ayudarte?


### Prueba 7 — Pedido INEXISTENTE

In [166]:
respuesta = preguntar_al_agente(
    'Quiero devolver el pedido ECO-99999. No me gusta el color.'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)


--- Pasos intermedios ---
  [HumanMessage]: Quiero devolver el pedido ECO-99999. No me gusta el color.
  [AIMessage]: 
  [ToolMessage]: {"pedido_id": "ECO-99999", "elegible": false, "mensaje": "No se encontró el pedido ECO-99999. Verifica el número e intenta nuevamente."}
  [AIMessage]: Lo siento mucho, pero parece que hubo un error al verificar la elegibilidad de tu devolución. El mensaje indica que no se encontró el pedido ECO-99999. ¿Podrías verificar el número de pedido y asegura
--- Fin pasos intermedios ---


RESPUESTA FINAL:
Lo siento mucho, pero parece que hubo un error al verificar la elegibilidad de tu devolución. El mensaje indica que no se encontró el pedido ECO-99999. ¿Podrías verificar el número de pedido y asegurarte de que es correcto? Si necesitas ayuda o tienes alguna otra pregunta, no dudes en preguntar. ¿Hay algo más en lo que pueda ayudarte?


### Prueba 8 — Consulta fuera de dominio

In [193]:
respuesta = preguntar_al_agente(
    '¿Me puedes recomendar una receta de pasta carbonara?'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)


--- Pasos intermedios ---
  [HumanMessage]: ¿Me puedes recomendar una receta de pasta carbonara?
  [AIMessage]: Lo siento, pero como asistente de EcoMarket, solo puedo ayudar con temas relacionados a pedidos, devoluciones, productos o servicios de nuestra tienda. La receta de pasta carbonara no está dentro de m
--- Fin pasos intermedios ---


RESPUESTA FINAL:
Lo siento, pero como asistente de EcoMarket, solo puedo ayudar con temas relacionados a pedidos, devoluciones, productos o servicios de nuestra tienda. La receta de pasta carbonara no está dentro de mi dominio de conocimiento. 

Si necesitas ayuda con algo relacionado a EcoMarket, estaré encantado de asistirte. De lo contrario, te recomiendo buscar en fuentes especializadas en cocina para obtener la receta que buscas.

¿Hay algo más en lo que pueda ayudarte?


### Prueba 9 — Intento de generar etiqueta sin verificar (anti-bypass)

In [194]:
# Limpiar estado para probar el mecanismo de protección
devoluciones_aprobadas.clear()

# Test determinista: llamar directamente a la herramienta sin aprobación previa
resultado_bypass = generar_etiqueta_devolucion.invoke({"pedido_id": "ECO-12350"})
print("--- Test directo del mecanismo anti-bypass ---")
print(f"Resultado: {resultado_bypass}")
print()

# Ahora probar vía agente: el agente debería verificar primero
respuesta = preguntar_al_agente(
    'Genera directamente la etiqueta de devolución para el pedido ECO-12350.'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)

--- Test directo del mecanismo anti-bypass ---
Resultado: {"pedido_id": "ECO-12350", "exito": false, "mensaje": "No se puede generar etiqueta: no hay devolución aprobada para ECO-12350. Verifica elegibilidad primero."}


--- Pasos intermedios ---
  [HumanMessage]: Genera directamente la etiqueta de devolución para el pedido ECO-12350.
  [AIMessage]: 
  [ToolMessage]: {"pedido_id": "ECO-12350", "exito": false, "mensaje": "No se puede generar etiqueta: no hay devolución aprobada para ECO-12350. Verifica elegibilidad primero."}
  [AIMessage]: Lo siento, no puedo generar la etiqueta de devolución directamente porque no hay una devolución aprobada previamente para el pedido ECO-12350. Primero, necesito verificar la elegibilidad de la devoluc
--- Fin pasos intermedios ---


RESPUESTA FINAL:
Lo siento, no puedo generar la etiqueta de devolución directamente porque no hay una devolución aprobada previamente para el pedido ECO-12350. Primero, necesito verificar la elegibilidad de la devolución.

### Revisión del Log de Acciones

In [169]:
log_file = LOG_PATH / 'agent_actions.jsonl'
if log_file.exists():
    with open(log_file, encoding='utf-8') as f:
        lineas = f.readlines()
    print(f'Total de acciones registradas: {len(lineas)}\n')
    for linea in lineas[-5:]:
        entrada = json.loads(linea)
        print(f"  [{entrada['timestamp']}] {entrada['herramienta']} | input: {entrada['input']}")
else:
    print('No hay acciones registradas aún.')

Total de acciones registradas: 119

  [2026-05-18T11:50:07.513076] verificar_elegibilidad_devolucion | input: {'pedido_id': 'ECO-12348', 'motivo': 'paquete roto y aplastado'}
  [2026-05-18T11:50:28.826490] verificar_elegibilidad_devolucion | input: {'pedido_id': 'ECO-99999', 'motivo': 'no me gusta el color'}
  [2026-05-18T11:50:49.057669] consultar_base_conocimiento | input: {'pregunta': 'receta de pasta carbonara'}
  [2026-05-18T11:51:11.268341] generar_etiqueta_devolucion | input: {'pedido_id': 'ECO-12350'}
  [2026-05-18T11:51:22.011759] verificar_elegibilidad_devolucion | input: {'pedido_id': 'ECO-12350', 'motivo': 'no me gusta el producto.'}


### Resumen de Evaluación

| # | Escenario | Herramientas invocadas | Resultado esperado |
|---|---|---|---|
| 1 | Estado de pedido (EN TRÁNSITO) | `consultar_estado_pedido` | Info de tracking |
| 2 | Devolución ELEGIBLE | `verificar` → `generar_etiqueta` | Instrucciones completas |
| 3 | Devolución NO ELEGIBLE (tránsito) | `verificar` | Negativa empática |
| 4 | Devolución CANCELADO | `verificar` | Negativa + alternativa |
| 5 | Devolución por DAÑO | `verificar` → `generar_etiqueta` | Aprobación + compensación |
| 6 | Consulta general de política | `consultar_base_conocimiento` | Info del RAG |
| 7 | Pedido INEXISTENTE | `verificar` | Error amable |
| 8 | Fuera de dominio | Ninguna | Rechazo amable |
| 9 | Bypass: etiqueta sin verificar | `generar_etiqueta` (falla) | Error controlado |

## Fase 4: Despliegue con Gradio

- `gr.ChatInterface` provee interfaz de chat lista para demostración.
- `share=True` en Colab despliega públicamente sin configuración adicional.
- La interfaz tipo chat es natural para demostrar el comportamiento del agente.

In [170]:
try:
    import gradio as gr
    print(f'Gradio disponible: v{gr.__version__}')
except ImportError:
    !pip install gradio -q
    import gradio as gr
    print(f'Gradio instalado: v{gr.__version__}')

Gradio disponible: v6.14.0


In [171]:
import gradio as gr

def responder(mensaje: str, historial: list):
    """Conecta Gradio con el agente."""
    if not mensaje.strip():
        yield "Por favor, escribe tu consulta para que pueda ayudarte."
        return
    respuesta = preguntar_al_agente(mensaje.strip(), verbose=False)
    yield respuesta

demo = gr.ChatInterface(
    fn=responder,
    title="🌱 EcoBot — Asistente de EcoMarket",
    description=(
        "Bienvenido al asistente inteligente de EcoMarket. Puedo ayudarte con:<br><br>"
        "• Estado de tus pedidos<br>"
        "• Proceso de devoluciones<br>"
        "• Preguntas sobre nuestra política de devoluciones<br><br>"
        "Ejemplos: <i>¿Cuál es el estado del pedido ECO-XXXXX?</i> · "
        "<i>Quiero devolver el pedido ECO-XXXXX</i> · "
        "<i>¿Cuántos días tengo para devolver?</i>"
    ),
    examples=[
        "¿Cuál es el estado de mi pedido ECO-12345?",
        "Quiero devolver mi pedido ECO-12347, no me gusta el tamaño.",
        "¿Cuántos días tengo para hacer una devolución?",
        "¿Cuáles son los números de contacto?",
        "¿Qué productos no se pueden devolver?"
    ],
)

demo.launch(share=IN_COLAB)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
